<a href="https://colab.research.google.com/github/vsolanki76771215/Student_MLE_MiniProject_EDA/blob/main/PrototypeScaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# STEP 1 — Scalable scikit-learn Prototype
# Uses incremental learning for large tabular data
# ============================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

DATA_PATH = "sample_data/scaled_deforestation_ml_dataset.csv"

feature_cols = [
    "ndvi_2018_mean",
    "ndvi_2022_mean",
    "ndvi_diff",
    "treecover_mean"
]

target_col = "loss_binary"

df = pd.read_csv(DATA_PATH)

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

sgd_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

sgd_pipeline.fit(X_train, y_train)

y_pred = sgd_pipeline.predict(X_test)
y_prob = sgd_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.87      0.89     26939
           1       0.41      0.51      0.46      4892

    accuracy                           0.81     31831
   macro avg       0.66      0.69      0.67     31831
weighted avg       0.83      0.81      0.82     31831

Confusion Matrix:
[[23381  3558]
 [ 2388  2504]]
ROC-AUC: 0.8066392775817033
PR-AUC: 0.35795040552536983


In [4]:
# ============================================================
# STEP 2 — Scalable Random Forest Baseline
# Useful comparison model for structured satellite features
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("ROC-AUC:", roc_auc_score(y_test, rf_prob))
print("PR-AUC:", average_precision_score(y_test, rf_prob))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.88      0.92     26939
           1       0.55      0.78      0.65      4892

    accuracy                           0.87     31831
   macro avg       0.75      0.83      0.78     31831
weighted avg       0.89      0.87      0.88     31831

Random Forest Confusion Matrix:
[[23826  3113]
 [ 1066  3826]]
ROC-AUC: 0.9263275510824446
PR-AUC: 0.7097944398118861


In [6]:
# ============================================================
# STEP 3 — TensorFlow / Keras Scalable Deep Learning Prototype
# Uses tf.data batching for larger datasets
# ============================================================

import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

df = pd.read_csv("sample_data/scaled_deforestation_ml_dataset.csv")

X = df[feature_cols].values.astype("float32")
y = df[target_col].values.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

BATCH_SIZE = 4096

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(50_000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            patience=4,
            mode="max",
            restore_best_weights=True
        )
    ]
)

y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.7257 - loss: 0.6601 - pr_auc: 0.3989 - roc_auc: 0.7858 - val_accuracy: 0.8478 - val_loss: 0.5792 - val_pr_auc: 0.4967 - val_roc_auc: 0.7387
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.7931 - loss: 0.5518 - pr_auc: 0.4738 - roc_auc: 0.8156 - val_accuracy: 0.8465 - val_loss: 0.5295 - val_pr_auc: 0.5278 - val_roc_auc: 0.7719
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.8255 - loss: 0.4862 - pr_auc: 0.4987 - roc_auc: 0.8257 - val_accuracy: 0.8466 - val_loss: 0.4789 - val_pr_auc: 0.5519 - val_roc_auc: 0.8260
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.8421 - loss: 0.4332 - pr_auc: 0.5085 - roc_auc: 0.8293 - val_accuracy: 0.8464 - val_loss: 0.4335 - val_pr_auc: 0.5521 - val_roc_auc: 0.8254
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.8542 - loss: 0.3905 - pr_auc: 0.5171 - roc_auc: 0.8370 - val_accuracy: 0.8466 - val_loss: 0.4002 - val_pr_auc: 0.55